# AyeAI Physical AI Platform - Manufacturing Backend Lab

**Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.**  
**AyeAI®, AyeAM® and AyeCNSe® are registered trademarks.**  
Licensed under **GNU General Public License v3.0 (GPL-3.0-only)**. See the package `LICENSE` and `THIRD-PARTY-NOTICES.md`.

This notebook is an optional browser-accessible execution environment for the Manufacturing drop-in. It does not replace the canonical platform graph, and it does not write to ERPNext, reserve factory capacity, or actuate machinery.

## Runtime truth

- OR-Tools CP-SAT: CPU solver; a Colab GPU does not accelerate CP-SAT.
- SciPy/HiGHS LP/MILP: CPU solver; a Colab GPU does not accelerate HiGHS.
- SimPy: discrete-event CPU simulation.
- PyTorch/JAX/CuPy GPU libraries are not substitutes for the above solvers. Use them only for workloads that their numerical models actually support.
- Solver output is a planning proposal; SimPy output is simulated evidence. Neither is physical verification.

Choose **Runtime → Change runtime type → T4 GPU** only when testing an independent GPU workload. CPU runtime is sufficient for the included manufacturing scheduling workers.


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks.
import sys, platform, json, os, subprocess, importlib.util
print("Python:", sys.version)
print("Platform:", platform.platform())
for name in ("numpy", "scipy", "ortools", "highspy", "simpy", "torch", "jax", "cupy"):
    print(f"{name:10s}", "AVAILABLE" if importlib.util.find_spec(name) else "not installed")
try:
    import torch
    print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("PyTorch probe:", type(exc).__name__, str(exc))
try:
    import jax
    print("JAX devices:", jax.devices())
except Exception as exc:
    print("JAX probe:", type(exc).__name__, str(exc))


## Install optional CPU backends

This installs optional numerical libraries into the temporary Colab runtime only. It does not alter the platform ZIP or the GitHub Pages browser application. Pin exact versions for production qualification; the ranges below are convenience constraints for an interactive lab.


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary.
%pip -q install "numpy>=2,<3" "scipy>=1.12,<2" "simpy>=4.1,<5"
# Optional CP-SAT backend. If installation fails, its worker must report UNAVAILABLE.
%pip -q install "ortools>=9.12,<10"


## Obtain the package

Upload `manufacturing-iteration48.zip` or the extracted `manufacturing/` folder to the Colab session. The ZIP must contain a top-level `manufacturing/` directory. The package is deliberately not fetched from an unverified URL.


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary.
from google.colab import files
uploaded = files.upload()  # Select the manufacturing package ZIP from your device.


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary.
import zipfile, pathlib, os
zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
if zip_names:
    with zipfile.ZipFile(zip_names[0]) as zf:
        zf.extractall("/content/pap")
PACKAGE = pathlib.Path("/content/pap/manufacturing")
assert PACKAGE.is_dir(), "Expected /content/pap/manufacturing after ZIP extraction."
print("Package files:", sum(1 for p in PACKAGE.rglob("*") if p.is_file()))


## Run the manufacturing package tests

These tests validate the package's own Node module and Python worker protocols. They do not establish factory qualification, solver performance at production scale, or live-system compatibility.


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary.
import subprocess, os
node = subprocess.run(["node", "--experimental-default-type=module", "test.mjs"],
                      cwd=PACKAGE, text=True, capture_output=True)
print(node.stdout)
if node.returncode:
    print(node.stderr)
    raise RuntimeError("Manufacturing Node tests failed")
worker = subprocess.run([sys.executable, "workers/test_workers.py"],
                        cwd=PACKAGE, text=True, capture_output=True)
print(worker.stdout)
if worker.returncode:
    print(worker.stderr)
    raise RuntimeError("Manufacturing worker tests failed")


## Execute a real finite-capacity job-shop case

The SciPy `milp` implementation uses the HiGHS optimization engine. This is CPU execution, even if the Colab runtime has a GPU. The model includes one machine with two jobs competing for finite capacity. The solver must return a non-overlapping schedule or an explicit non-success status.


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary.
import json, subprocess, sys
model = {
  "modelVersion":"1.0.0", "timeUnit":"minute", "horizonMinutes":12,
  "operations":[
    {"id":"op-a","orderId":"job-a","sequence":0,"process":"MILL","durationMinutes":7,"eligibleStationIds":["mill-01"]},
    {"id":"op-b","orderId":"job-b","sequence":0,"process":"MILL","durationMinutes":5,"eligibleStationIds":["mill-01"]}
  ],
  "stations":[{"id":"mill-01","availableMinutes":12}]
}
request = {
  "protocol":"ayeai.manufacturing.backend/v1",
  "requestId":"colab-jobshop-001",
  "operation":"solve-jobshop",
  "payload":{"model":model,"solverOptions":{"timeLimitSeconds":30}}
}
worker = PACKAGE/"workers"/"scipy_highs_jobshop_worker.py"
proc = subprocess.run([sys.executable,str(worker)],input=json.dumps(request),
                      text=True,capture_output=True,timeout=60)
if proc.returncode:
    raise RuntimeError(proc.stderr)
response = json.loads(proc.stdout)
print(json.dumps(response,indent=2))
if response["status"] == "SUCCEEDED":
    a = response["result"]["assignments"]
    assert len(a) == 2
    assert (a[0]["endMinute"] <= a[1]["startMinute"] + 1e-7 or
            a[1]["endMinute"] <= a[0]["startMinute"] + 1e-7)
    assert abs(response["result"]["makespanMinutes"] - 12) < 1e-6
else:
    raise RuntimeError("Expected a feasible 12-minute schedule")


## Optional GPU availability probe

The following cell only detects a GPU. It does not claim the manufacturing solvers use it. The separate `compute-orchestration` package contains a GPU-capable Monte Carlo reliability workload that uses PyTorch CUDA when available and falls back to NumPy CPU execution.


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary.
try:
    import torch
    gpu = torch.cuda.is_available()
    print("PyTorch CUDA:", gpu)
    if gpu:
        print("Device:", torch.cuda.get_device_name(0))
    else:
        print("No CUDA GPU attached. CPU execution remains valid.")
except ImportError:
    print("PyTorch not installed; no GPU capability is claimed.")


## Export evidence and results

Keep the JSON result with its request ID, backend version, solver status, model inputs and diagnostics. Treat it as a planning artifact. The canonical platform must bind it to the correct system configuration and evidence lifecycle before any promotion.
